# 04 — Feature Engineering & Modeling Dataset Construction

## Healthcare 30-Day Readmission Analytics

This notebook converts the findings from exploratory data analysis into a reproducible, leakage-aware feature-engineering pipeline for predicting 30-day hospital readmission.

The objective is not simply to transform variables into model-compatible representations. Feature construction must preserve the clinical meaning of the underlying data, respect the intended prediction timestamp, prevent information leakage, and ensure that preprocessing decisions are learned exclusively from training data before being applied unchanged to held-out observations.

The workflow therefore emphasizes:

- explicit definition of the prediction problem and unit of analysis;
- identification and exclusion of identifiers and outcome-derived variables;
- patient-aware validation safeguards where repeated encounters exist;
- clinically interpretable feature engineering;
- consolidation of sparse categorical variables using training-derived rules;
- appropriate treatment of numeric, binary, ordinal, and nominal predictors;
- reproducible preprocessing pipelines;
- preservation of an untouched test set for final model evaluation;
- validation of the final feature matrices before predictive modeling.

The output of this notebook will be a modeling-ready representation of the hospital encounter data suitable for baseline and advanced classification models.

## 4A — Modeling Objective, Unit of Analysis & Prediction Timestamp

Before constructing predictors, the prediction problem must be defined precisely.

### Prediction objective

The modeling task is binary classification:

> **Predict whether an eligible hospital encounter will be followed by readmission within 30 days.**

The target variable is `readmitted_30d`, where:

- `1` indicates readmission within 30 days;
- `0` indicates no documented readmission within 30 days.

### Unit of analysis

The primary unit of analysis is the **hospital encounter** rather than the unique patient.

Because individual patients may contribute multiple encounters, observations are not necessarily statistically independent. Patient identifiers must therefore never be used as predictive features, and repeated-patient structure must be considered when constructing validation and test partitions.

### Prediction timestamp

The intended prediction timestamp is defined as **at or near the end of the index hospitalization, before the future 30-day readmission outcome is known**.

A predictor is eligible only if it would reasonably be available at that prediction point.

This distinction is critical because variables recorded after the prediction decision, variables derived from future encounters, and direct transformations of the target would constitute information leakage.

Feature eligibility will therefore be evaluated according to:

1. whether the variable is available by the prediction timestamp;
2. whether it contains information derived from the future outcome;
3. whether it functions primarily as an identifier rather than a predictor;
4. whether its clinical meaning is appropriate for the intended use case.

## 4B — Load Processed Data & Reproducibility Setup

The feature-engineering workflow begins from the processed dataset produced by the preprocessing stage rather than from the original raw data.

This preserves separation of responsibilities across the project:

- Notebook 01 audits the source data;
- Notebook 02 performs deterministic preprocessing and data cleaning;
- Notebook 03 performs training-set exploratory analysis;
- Notebook 04 constructs modeling-ready features.

All stochastic operations use an explicit random seed to support reproducibility.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GroupShuffleSplit,
    train_test_split,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.impute import SimpleImputer

# Reproducibility
RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

print(f"Random state: {RANDOM_STATE}")
print(f"Pandas version: {pd.__version__}")

Random state: 42
Pandas version: 3.0.5


In [2]:
# Resolve project root from the notebook working directory

cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
INTERIM_DIR = DATA_DIR / "interim"

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Processed directory exists: {PROCESSED_DIR.exists()}")

Project root: c:\Users\amohe.000\Downloads\healthcare-readmission-analytics
Processed data directory: c:\Users\amohe.000\Downloads\healthcare-readmission-analytics\data\processed
Processed directory exists: True


In [3]:
# Inspect processed datasets produced by earlier pipeline stages

processed_files = sorted(path for path in PROCESSED_DIR.glob("*") if path.is_file())

print("Available processed-data artifacts")
print("-" * 60)

for path in processed_files:
    size_mb = path.stat().st_size / (1024**2)
    print(f"{path.name:<45} {size_mb:>8.2f} MB")

Available processed-data artifacts
------------------------------------------------------------
feature_registry.json                             0.00 MB
test.csv                                          3.62 MB
train.csv                                        17.02 MB
validation.csv                                    3.57 MB


## 4C — Load Modeling Splits & Leakage Audit

The preprocessing pipeline has already produced dedicated training, validation, and test datasets. These partitions are preserved rather than reconstructed in this notebook.

This is methodologically important because downstream feature engineering must respect the established data-splitting boundary. Any transformation that learns information from the data—including imputation parameters, scaling parameters, category frequencies, sparse-category mappings, and encoding vocabularies—must be fit using the training partition only.

The validation partition will support model development and model-selection decisions, while the test partition will remain untouched until final model evaluation.

Before engineering predictors, this section verifies:

- split dimensions and schema consistency;
- target prevalence across partitions;
- identifier structure;
- repeated-patient overlap across partitions;
- potential outcome-derived variables;
- feature metadata recorded by the preprocessing pipeline; and
- candidate variables that require explicit leakage or prediction-timestamp decisions.

In [4]:
# Load processed datasets for modeling - Load all three splits
TRAIN_PATH = PROCESSED_DIR / "train.csv"
VALIDATION_PATH = PROCESSED_DIR / "validation.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"

df_train = pd.read_csv(TRAIN_PATH)
df_validation = pd.read_csv(VALIDATION_PATH)
df_test = pd.read_csv(TEST_PATH)

print("Processed modeling splits")
print("-" * 60)
print(f"Train:      {df_train.shape}")
print(f"Validation: {df_validation.shape}")
print(f"Test:       {df_test.shape}")

print()
print(f"Total encounters: {len(df_train) + len(df_validation) + len(df_test):,}")

Processed modeling splits
------------------------------------------------------------
Train:      (71520, 48)
Validation: (15027, 48)
Test:       (15219, 48)

Total encounters: 101,766


In [5]:
# Verify that all processed splits expose the same schema

train_cols = list(df_train.columns)
validation_cols = list(df_validation.columns)
test_cols = list(df_test.columns)

print("Schema consistency")
print("-" * 60)

print(f"Train columns:      {len(train_cols)}")
print(f"Validation columns: {len(validation_cols)}")
print(f"Test columns:       {len(test_cols)}")

print()
print(f"Train == Validation: {train_cols == validation_cols}")
print(f"Train == Test:       {train_cols == test_cols}")

train_only = sorted(set(train_cols) - set(validation_cols) - set(test_cols))
validation_only = sorted(set(validation_cols) - set(train_cols))
test_only = sorted(set(test_cols) - set(train_cols))

print()
print(f"Train-only columns:      {train_only}")
print(f"Validation-only columns: {validation_only}")
print(f"Test-only columns:       {test_only}")

Schema consistency
------------------------------------------------------------
Train columns:      48
Validation columns: 48
Test columns:       48

Train == Validation: True
Train == Test:       True

Train-only columns:      []
Validation-only columns: []
Test-only columns:       []


In [6]:
# Load feature registry
import json

FEATURE_REGISTRY_PATH = PROCESSED_DIR / "feature_registry.json"

with open(FEATURE_REGISTRY_PATH, "r", encoding="utf-8") as f:
    feature_registry = json.load(f)

print("Feature registry")
print("-" * 60)
print(f"Registry type: {type(feature_registry).__name__}")

if isinstance(feature_registry, dict):
    print(f"Top-level keys: {list(feature_registry.keys())}")
else:
    print(f"Registry entries: {len(feature_registry)}")

Feature registry
------------------------------------------------------------
Registry type: dict
Top-level keys: ['target', 'source_outcome', 'identifiers', 'model_features', 'quantitative_features', 'categorical_features']


In [7]:
# Verify that the target variable is present in all splits (Target integrity across splits)
TARGET = "readmitted_30d"

assert TARGET in df_train.columns
assert TARGET in df_validation.columns
assert TARGET in df_test.columns


def summarize_target(df, split_name):
    counts = df[TARGET].value_counts(dropna=False).sort_index()
    prevalence = df[TARGET].mean() * 100

    print(f"{split_name}")
    print("-" * 40)
    print(f"Encounters:          {len(df):,}")
    print(f"Positive outcomes:   {int(df[TARGET].sum()):,}")
    print(f"Readmission rate:    {prevalence:.2f}%")
    print(f"Missing target:      {df[TARGET].isna().sum():,}")
    print(f"Observed values:     {sorted(df[TARGET].dropna().unique().tolist())}")
    print()


for split_name, df in [
    ("Train", df_train),
    ("Validation", df_validation),
    ("Test", df_test),
]:
    summarize_target(df, split_name)

Train
----------------------------------------
Encounters:          71,520
Positive outcomes:   8,021
Readmission rate:    11.22%
Missing target:      0
Observed values:     [0, 1]

Validation
----------------------------------------
Encounters:          15,027
Positive outcomes:   1,656
Readmission rate:    11.02%
Missing target:      0
Observed values:     [0, 1]

Test
----------------------------------------
Encounters:          15,219
Positive outcomes:   1,680
Readmission rate:    11.04%
Missing target:      0
Observed values:     [0, 1]



In [8]:
# Verify that the identifier columns are present in all splits - Identifier integrity across splits (Identifier and patient-overlap audit)
ENCOUNTER_ID = "encounter_id"
PATIENT_ID = "patient_nbr"

for col in [ENCOUNTER_ID, PATIENT_ID]:
    assert col in df_train.columns
    assert col in df_validation.columns
    assert col in df_test.columns

print("Identifier structure")
print("-" * 60)

for split_name, df in [
    ("Train", df_train),
    ("Validation", df_validation),
    ("Test", df_test),
]:
    print(
        f"{split_name:<10} | "
        f"encounters={len(df):>7,} | "
        f"unique encounter IDs={df[ENCOUNTER_ID].nunique():>7,} | "
        f"unique patients={df[PATIENT_ID].nunique():>7,}"
    )

train_patients = set(df_train[PATIENT_ID].dropna())
validation_patients = set(df_validation[PATIENT_ID].dropna())
test_patients = set(df_test[PATIENT_ID].dropna())

print()
print("Patient overlap across partitions")
print("-" * 60)
print(f"Train ∩ Validation: {len(train_patients & validation_patients):,}")
print(f"Train ∩ Test:       {len(train_patients & test_patients):,}")
print(f"Validation ∩ Test:  {len(validation_patients & test_patients):,}")

Identifier structure
------------------------------------------------------------
Train      | encounters= 71,520 | unique encounter IDs= 71,520 | unique patients= 50,062
Validation | encounters= 15,027 | unique encounter IDs= 15,027 | unique patients= 10,728
Test       | encounters= 15,219 | unique encounter IDs= 15,219 | unique patients= 10,728

Patient overlap across partitions
------------------------------------------------------------
Train ∩ Validation: 0
Train ∩ Test:       0
Validation ∩ Test:  0


In [9]:
# Search schema for variables that may encode or derive from the outcome

leakage_keywords = [
    "readmit",
    "target",
    "outcome",
    "label",
]

leakage_candidates = [
    col
    for col in df_train.columns
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

print("Potential outcome / leakage-related columns")
print("-" * 60)

for col in leakage_candidates:
    print(
        f"{col:<35} "
        f"dtype={str(df_train[col].dtype):<12} "
        f"unique={df_train[col].nunique(dropna=False):>5}"
    )

Potential outcome / leakage-related columns
------------------------------------------------------------
readmitted                          dtype=str          unique=    3
readmitted_30d                      dtype=int64        unique=    2


## 4D — Feature Inventory & Predictor Taxonomy

Feature engineering begins with an explicit classification of available variables according to their role in the modeling problem.

Predictors are separated from:

- the binary modeling target;
- the original outcome representation from which the target was derived;
- encounter identifiers;
- patient identifiers used only for grouping and validation safeguards; and
- variables requiring additional prediction-timestamp or leakage review.

This separation prevents identifiers and outcome-derived information from entering the predictive feature matrix accidentally.

Candidate predictors are subsequently classified by analytical role so that numeric, binary, categorical, clinical, utilization, and engineered variables can receive appropriate transformations.

Importantly, feature-selection and transformation decisions are derived from the training partition only. Validation and test data remain held out from all fitting decisions.

In [10]:
# Define columns that must never enter the predictive feature matrix

TARGET = "readmitted_30d"

IDENTIFIER_COLUMNS = [
    "encounter_id",
    "patient_nbr",
]

OUTCOME_DERIVED_COLUMNS = [
    "readmitted",
]

MANDATORY_EXCLUSIONS = IDENTIFIER_COLUMNS + OUTCOME_DERIVED_COLUMNS + [TARGET]

print("Mandatory modeling exclusions")
print("-" * 60)

for col in MANDATORY_EXCLUSIONS:
    status = "present" if col in df_train.columns else "not present"
    print(f"{col:<30} {status}")

Mandatory modeling exclusions
------------------------------------------------------------
encounter_id                   present
patient_nbr                    present
readmitted                     present
readmitted_30d                 present


In [11]:
# Separate targets from candidate predictors while preserving held-out splits

y_train = df_train[TARGET].copy()
y_validation = df_validation[TARGET].copy()
y_test = df_test[TARGET].copy()

X_train_raw = df_train.drop(columns=MANDATORY_EXCLUSIONS).copy()
X_validation_raw = df_validation.drop(columns=MANDATORY_EXCLUSIONS).copy()
X_test_raw = df_test.drop(columns=MANDATORY_EXCLUSIONS).copy()

print("Initial modeling matrices")
print("-" * 60)

print(f"X_train_raw:      {X_train_raw.shape}")
print(f"X_validation_raw: {X_validation_raw.shape}")
print(f"X_test_raw:       {X_test_raw.shape}")

print()
print(f"y_train:          {y_train.shape}")
print(f"y_validation:     {y_validation.shape}")
print(f"y_test:           {y_test.shape}")

assert list(X_train_raw.columns) == list(X_validation_raw.columns)
assert list(X_train_raw.columns) == list(X_test_raw.columns)

assert not any(col in X_train_raw.columns for col in MANDATORY_EXCLUSIONS)

print()
print("✓ Predictor schemas are aligned.")
print("✓ Mandatory exclusions are absent from predictor matrices.")

Initial modeling matrices
------------------------------------------------------------
X_train_raw:      (71520, 44)
X_validation_raw: (15027, 44)
X_test_raw:       (15219, 44)

y_train:          (71520,)
y_validation:     (15027,)
y_test:           (15219,)

✓ Predictor schemas are aligned.
✓ Mandatory exclusions are absent from predictor matrices.


In [12]:
# Build a compact inventory of candidate predictor structure

feature_inventory = pd.DataFrame(
    {
        "feature": X_train_raw.columns,
        "dtype": X_train_raw.dtypes.astype(str).values,
        "missing_n": X_train_raw.isna().sum().values,
        "missing_pct": (X_train_raw.isna().mean().values * 100),
        "n_unique": [
            X_train_raw[col].nunique(dropna=False) for col in X_train_raw.columns
        ],
    }
)

feature_inventory["missing_pct"] = feature_inventory["missing_pct"].round(2)

feature_inventory = feature_inventory.sort_values(["dtype", "feature"]).reset_index(
    drop=True
)

print(f"Candidate predictors: {len(feature_inventory)}")

feature_inventory

Candidate predictors: 44


,feature,dtype,missing_n,missing_pct,n_unique
0,num_lab_procedures,int64,0,0.0,115
1,num_medications,int64,0,0.0,74
2,num_procedures,int64,0,0.0,7
3,number_diagnoses,int64,0,0.0,16
4,number_emergency,int64,0,0.0,32
5,number_inpatient,int64,0,0.0,21
6,number_outpatient,int64,0,0.0,34
7,time_in_hospital,int64,0,0.0,14
8,A1Cresult,str,0,0.0,4
9,acarbose,str,0,0.0,4


In [13]:
# Initial structural classification based on stored data types

numeric_candidates = X_train_raw.select_dtypes(include=["number"]).columns.tolist()

categorical_candidates = X_train_raw.select_dtypes(exclude=["number"]).columns.tolist()

print("Initial predictor taxonomy")
print("-" * 60)

print(f"Numeric candidates:     {len(numeric_candidates)}")
print(f"Categorical candidates: {len(categorical_candidates)}")
print(
    f"Total:                  {len(numeric_candidates) + len(categorical_candidates)}"
)

print("\nNumeric candidates:")
for col in numeric_candidates:
    print(f"  - {col}")

print("\nCategorical candidates:")
for col in categorical_candidates:
    print(f"  - {col}")

Initial predictor taxonomy
------------------------------------------------------------
Numeric candidates:     8
Categorical candidates: 36
Total:                  44

Numeric candidates:
  - time_in_hospital
  - num_lab_procedures
  - num_procedures
  - num_medications
  - number_outpatient
  - number_emergency
  - number_inpatient
  - number_diagnoses

Categorical candidates:
  - race
  - gender
  - age
  - payer_code
  - medical_specialty
  - diag_1
  - diag_2
  - diag_3
  - max_glu_serum
  - A1Cresult
  - metformin
  - repaglinide
  - nateglinide
  - chlorpropamide
  - glimepiride
  - acetohexamide
  - glipizide
  - glyburide
  - tolbutamide
  - pioglitazone
  - rosiglitazone
  - acarbose
  - miglitol
  - troglitazone
  - tolazamide
  - insulin
  - glyburide-metformin
  - glipizide-metformin
  - glimepiride-pioglitazone
  - metformin-rosiglitazone
  - metformin-pioglitazone
  - change
  - diabetesMed
  - admission_type_description
  - discharge_disposition_description
  - admissio

## 4E — Feature Provenance & Semantic Classification

Storage data type alone is insufficient for determining an appropriate modeling transformation.

For example, integer-valued variables may represent:

- true quantitative counts;
- ordinal categories;
- binary indicators;
- administrative identifiers; or
- encoded nominal categories.

Similarly, some predictors in the processed dataset were engineered during earlier analytical stages and should be distinguished from their raw source variables.

This section therefore examines feature provenance and semantic role before defining the transformation pipeline. The goal is to preserve clinically meaningful structure while avoiding arbitrary transformations based solely on technical data type.

In [14]:
# Display candidate features in their modeling order

print("Candidate predictor features")
print("-" * 70)

for i, col in enumerate(X_train_raw.columns, start=1):
    print(
        f"{i:>2}. {col:<40} "
        f"dtype={str(X_train_raw[col].dtype):<10} "
        f"unique={X_train_raw[col].nunique(dropna=False):>5}"
    )

Candidate predictor features
----------------------------------------------------------------------
 1. race                                     dtype=str        unique=    6
 2. gender                                   dtype=str        unique=    3
 3. age                                      dtype=str        unique=   10
 4. time_in_hospital                         dtype=int64      unique=   14
 5. payer_code                               dtype=str        unique=   17
 6. medical_specialty                        dtype=str        unique=   70
 7. num_lab_procedures                       dtype=int64      unique=  115
 8. num_procedures                           dtype=int64      unique=    7
 9. num_medications                          dtype=int64      unique=   74
10. number_outpatient                        dtype=int64      unique=   34
11. number_emergency                         dtype=int64      unique=   32
12. number_inpatient                         dtype=int64      unique=   21


## 4F — Semantic Feature Registry & Modeling Eligibility

The candidate feature inventory contains both original variables and engineered representations created during preprocessing and exploratory analysis. Including every available representation indiscriminately could introduce unnecessary dimensionality, redundancy, unstable sparse categories, or information that is inappropriate for the intended prediction timestamp.

Feature eligibility is therefore determined according to four principles:

1. **Clinical interpretability** — retained variables should represent meaningful patient, utilization, diagnostic, treatment, or encounter characteristics.
2. **Dimensional stability** — extremely high-cardinality raw categorical variables should be replaced by clinically meaningful consolidated representations when available.
3. **Redundancy control** — raw and engineered versions of the same underlying concept should not automatically be included together.
4. **Prediction-time availability** — variables must be available at the intended prediction timestamp and must not encode future outcome information.

The resulting feature registry provides an explicit and reproducible contract for downstream preprocessing and modeling.

In [15]:
# Identify engineered and clinically consolidated features available to modeling

engineered_keywords = [
    "family",
    "group",
    "burden",
    "total",
    "description",
]

engineered_candidates = [
    col
    for col in X_train_raw.columns
    if any(keyword in col.lower() for keyword in engineered_keywords)
]

print("Engineered / consolidated feature candidates")
print("-" * 70)

for col in engineered_candidates:
    print(
        f"{col:<40} "
        f"dtype={str(X_train_raw[col].dtype):<10} "
        f"unique={X_train_raw[col].nunique(dropna=False):>5}"
    )

Engineered / consolidated feature candidates
----------------------------------------------------------------------
admission_type_description               dtype=str        unique=    8
discharge_disposition_description        dtype=str        unique=   26
admission_source_description             dtype=str        unique=   17


## 4G — High-Cardinality Feature Review

Nominal categorical variables with many distinct levels can substantially expand a one-hot encoded design matrix and may create unstable coefficients for rare categories.

This issue is especially relevant to raw diagnosis codes and medical-specialty fields. Where clinically meaningful consolidated representations are available, those representations are preferred over hundreds of sparse raw categories.

Cardinality is therefore reviewed before constructing the preprocessing pipeline.

In [16]:
# Review categorical feature cardinality

categorical_cardinality = (
    pd.DataFrame(
        {
            "feature": categorical_candidates,
            "n_unique": [
                X_train_raw[col].nunique(dropna=False) for col in categorical_candidates
            ],
        }
    )
    .sort_values("n_unique", ascending=False)
    .reset_index(drop=True)
)

categorical_cardinality["cardinality_class"] = pd.cut(
    categorical_cardinality["n_unique"],
    bins=[0, 2, 10, 50, float("inf")],
    labels=[
        "binary",
        "low",
        "moderate",
        "high",
    ],
    include_lowest=True,
)

categorical_cardinality

,feature,n_unique,cardinality_class
0,diag_3,731,high
1,diag_2,708,high
2,diag_1,692,high
3,medical_specialty,70,high
4,discharge_disposition_description,26,moderate
5,admission_source_description,17,moderate
6,payer_code,17,moderate
7,age,10,low
8,admission_type_description,8,low
9,race,6,low


In [17]:
# Inspect diagnosis-related features actually available in the modeling dataset

diagnosis_related_columns = [
    col
    for col in X_train_raw.columns
    if ("diag" in col.lower() or "diagnos" in col.lower())
]

print("Diagnosis-related modeling features")
print("-" * 70)

for col in diagnosis_related_columns:
    print(
        f"{col:<40} "
        f"dtype={str(X_train_raw[col].dtype):<10} "
        f"unique={X_train_raw[col].nunique(dropna=False):>5}"
    )

Diagnosis-related modeling features
----------------------------------------------------------------------
diag_1                                   dtype=str        unique=  692
diag_2                                   dtype=str        unique=  708
diag_3                                   dtype=str        unique=  731
number_diagnoses                         dtype=int64      unique=   16


## 4G.1 — Clinical Diagnosis Family Engineering

The processed modeling datasets retain the original ICD-9 diagnosis codes but do not persist the broader clinical-family variables created during exploratory analysis.

Rather than one-hot encoding hundreds of sparse raw diagnosis codes, Notebook 04 reconstructs clinically interpretable ICD-9 families deterministically for each diagnosis position.

Because the mapping is rule-based and does not learn from outcome frequencies or held-out data, the same transformation can be applied identically to training, validation, and test partitions without introducing leakage.

In [18]:
# Map ICD-9 diagnosis codes to broad clinical families


def map_icd9_family(code):
    if pd.isna(code):
        return "Unknown"

    code = str(code).strip()

    if code in {"?", "", "nan", "None"}:
        return "Unknown"

    # ICD-9 V and E codes
    if code.startswith("V"):
        return "Supplementary factors"

    if code.startswith("E"):
        return "External causes"

    # Use the integer portion before any decimal
    try:
        numeric_code = int(float(code))
    except ValueError:
        return "Other / Unclassified"

    if 1 <= numeric_code <= 139:
        return "Infectious & parasitic"
    elif 140 <= numeric_code <= 239:
        return "Neoplasms"
    elif 240 <= numeric_code <= 279:
        return "Endocrine, nutritional & metabolic"
    elif 280 <= numeric_code <= 289:
        return "Blood disorders"
    elif 290 <= numeric_code <= 319:
        return "Mental disorders"
    elif 320 <= numeric_code <= 389:
        return "Nervous system & sense organs"
    elif 390 <= numeric_code <= 459:
        return "Circulatory"
    elif 460 <= numeric_code <= 519:
        return "Respiratory"
    elif 520 <= numeric_code <= 579:
        return "Digestive"
    elif 580 <= numeric_code <= 629:
        return "Genitourinary"
    elif 630 <= numeric_code <= 679:
        return "Pregnancy & childbirth"
    elif 680 <= numeric_code <= 709:
        return "Skin & subcutaneous tissue"
    elif 710 <= numeric_code <= 739:
        return "Musculoskeletal & connective tissue"
    elif 740 <= numeric_code <= 759:
        return "Congenital anomalies"
    elif 760 <= numeric_code <= 779:
        return "Perinatal conditions"
    elif 780 <= numeric_code <= 799:
        return "Symptoms & ill-defined conditions"
    elif 800 <= numeric_code <= 999:
        return "Injury & poisoning"
    else:
        return "Other / Unclassified"

In [19]:
UTILIZATION_COLUMNS = [
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
]

DIAGNOSIS_COLUMNS = [
    "diag_1",
    "diag_2",
    "diag_3",
]


def engineer_clinical_features(df):
    """
    Create deterministic clinically motivated features.

    No parameters are estimated from the data, so the transformation can be
    applied identically to train, validation, and test partitions.
    """
    result = df.copy()

    # Prior healthcare-utilization aggregate
    result["prior_utilization_total"] = result[UTILIZATION_COLUMNS].sum(axis=1)

    # Right-skew transformations
    for col in UTILIZATION_COLUMNS:
        result[f"log1p_{col}"] = np.log1p(result[col])

    result["log1p_prior_utilization_total"] = np.log1p(
        result["prior_utilization_total"]
    )

    # Broad ICD-9 clinical families
    for col in DIAGNOSIS_COLUMNS:
        result[f"{col}_family"] = result[col].apply(map_icd9_family)

    return result

In [20]:
X_train_engineered = engineer_clinical_features(X_train_raw)
X_validation_engineered = engineer_clinical_features(X_validation_raw)
X_test_engineered = engineer_clinical_features(X_test_raw)

assert list(X_train_engineered.columns) == list(X_validation_engineered.columns)
assert list(X_train_engineered.columns) == list(X_test_engineered.columns)

new_features = [
    col for col in X_train_engineered.columns if col not in X_train_raw.columns
]

print("Engineered feature results")
print("-" * 60)
print(f"Before engineering: {X_train_raw.shape[1]}")
print(f"After engineering:  {X_train_engineered.shape[1]}")
print(f"Added features:      {len(new_features)}")

print("\nNew features:")
for col in new_features:
    print(f"  - {col}")

Engineered feature results
------------------------------------------------------------
Before engineering: 44
After engineering:  52
Added features:      8

New features:
  - prior_utilization_total
  - log1p_number_outpatient
  - log1p_number_emergency
  - log1p_number_inpatient
  - log1p_prior_utilization_total
  - diag_1_family
  - diag_2_family
  - diag_3_family


In [21]:
# Validate diagnosis-family construction

DIAGNOSIS_FAMILY_COLUMNS = [
    "diag_1_family",
    "diag_2_family",
    "diag_3_family",
]

print("Diagnosis-family validation")
print("-" * 70)

for col in DIAGNOSIS_FAMILY_COLUMNS:
    assert col in X_train_engineered.columns
    assert X_train_engineered[col].isna().sum() == 0

    print(f"{col:<20} " f"unique={X_train_engineered[col].nunique(dropna=False):>3}")

print("\n✓ All three diagnosis-family variables were created successfully.")

Diagnosis-family validation
----------------------------------------------------------------------
diag_1_family        unique= 18
diag_2_family        unique= 19
diag_3_family        unique= 19

✓ All three diagnosis-family variables were created successfully.


In [22]:
PRIMARY_MODEL_EXCLUSIONS = [
    "diag_1",
    "diag_2",
    "diag_3",
    "discharge_disposition_description",
]

X_train_model = X_train_engineered.drop(columns=PRIMARY_MODEL_EXCLUSIONS).copy()

X_validation_model = X_validation_engineered.drop(
    columns=PRIMARY_MODEL_EXCLUSIONS
).copy()

X_test_model = X_test_engineered.drop(columns=PRIMARY_MODEL_EXCLUSIONS).copy()

assert list(X_train_model.columns) == list(X_validation_model.columns)
assert list(X_train_model.columns) == list(X_test_model.columns)

print("Primary modeling feature set")
print("-" * 60)
print(f"Engineered features available: {X_train_engineered.shape[1]}")
print(f"Excluded from primary model:   {len(PRIMARY_MODEL_EXCLUSIONS)}")
print(f"Final predictors:              {X_train_model.shape[1]}")

Primary modeling feature set
------------------------------------------------------------
Engineered features available: 52
Excluded from primary model:   4
Final predictors:              48


## 4J — Final Feature Taxonomy

Following deterministic clinical feature engineering and prediction-timestamp exclusions, the primary modeling dataset contains the predictors eligible for downstream model development.

The remaining variables must now be assigned to preprocessing pathways according to their semantic role. This distinction is important because storage data type does not necessarily imply modeling type: integer-valued fields may represent quantitative counts, while string-valued fields may represent binary, ordinal, or nominal categories.

For the primary modeling pipeline:

- quantitative count and utilization variables are treated as numeric;
- categorical and treatment-status variables are encoded categorically;
- diagnosis-family variables replace the high-cardinality raw diagnosis codes;
- engineered logarithmic utilization measures remain numeric; and
- all preprocessing decisions are defined from the training schema and subsequently applied unchanged to validation and test data.

In [23]:
# Define semantically numeric predictors

NUMERIC_FEATURES = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "prior_utilization_total",
    "log1p_number_outpatient",
    "log1p_number_emergency",
    "log1p_number_inpatient",
    "log1p_prior_utilization_total",
]

# Retain only features actually present
NUMERIC_FEATURES = [col for col in NUMERIC_FEATURES if col in X_train_model.columns]

print(f"Numeric modeling features: {len(NUMERIC_FEATURES)}")
print("-" * 60)

for col in NUMERIC_FEATURES:
    print(
        f"{col:<35} "
        f"dtype={str(X_train_model[col].dtype):<10} "
        f"unique={X_train_model[col].nunique(dropna=False):>5}"
    )

Numeric modeling features: 13
------------------------------------------------------------
time_in_hospital                    dtype=int64      unique=   14
num_lab_procedures                  dtype=int64      unique=  115
num_procedures                      dtype=int64      unique=    7
num_medications                     dtype=int64      unique=   74
number_outpatient                   dtype=int64      unique=   34
number_emergency                    dtype=int64      unique=   32
number_inpatient                    dtype=int64      unique=   21
number_diagnoses                    dtype=int64      unique=   16
prior_utilization_total             dtype=int64      unique=   45
log1p_number_outpatient             dtype=float64    unique=   34
log1p_number_emergency              dtype=float64    unique=   32
log1p_number_inpatient              dtype=float64    unique=   21
log1p_prior_utilization_total       dtype=float64    unique=   45


In [24]:
# Assign all remaining primary predictors to the categorical pathway

CATEGORICAL_FEATURES = [
    col for col in X_train_model.columns if col not in NUMERIC_FEATURES
]

print("Final semantic feature taxonomy")
print("-" * 60)

print(f"Numeric features:     {len(NUMERIC_FEATURES)}")
print(f"Categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"Total:                {len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)}")

assert len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES) == X_train_model.shape[1]

assert set(NUMERIC_FEATURES).isdisjoint(CATEGORICAL_FEATURES)

print("\n✓ Every primary predictor belongs to exactly one preprocessing pathway.")

Final semantic feature taxonomy
------------------------------------------------------------
Numeric features:     13
Categorical features: 35
Total:                48

✓ Every primary predictor belongs to exactly one preprocessing pathway.


In [25]:
# Review final categorical cardinality before encoding

final_categorical_cardinality = (
    pd.DataFrame(
        {
            "feature": CATEGORICAL_FEATURES,
            "n_unique": [
                X_train_model[col].nunique(dropna=False) for col in CATEGORICAL_FEATURES
            ],
            "missing_n": [
                X_train_model[col].isna().sum() for col in CATEGORICAL_FEATURES
            ],
        }
    )
    .sort_values(["n_unique", "feature"], ascending=[False, True])
    .reset_index(drop=True)
)

final_categorical_cardinality["missing_pct"] = (
    final_categorical_cardinality["missing_n"] / len(X_train_model) * 100
).round(2)

final_categorical_cardinality

,feature,n_unique,missing_n,missing_pct
0,medical_specialty,70,0,0.0
1,diag_2_family,19,0,0.0
2,diag_3_family,19,0,0.0
3,diag_1_family,18,0,0.0
4,admission_source_description,17,0,0.0
5,payer_code,17,0,0.0
6,age,10,0,0.0
7,admission_type_description,8,0,0.0
8,race,6,0,0.0
9,A1Cresult,4,0,0.0


## 4K — Training-Derived Rare-Category Consolidation

Several nominal predictors contain categories represented by relatively few training encounters. Direct one-hot encoding of very sparse categories can produce unstable coefficients, unnecessarily expand the design matrix, and increase sensitivity to idiosyncratic observations.

Rare-category consolidation is therefore learned exclusively from the training partition.

For each categorical feature, categories occurring in fewer than a predefined number of training encounters are mapped to `Other`. The resulting training-derived category sets are then applied unchanged to validation and test data. Categories appearing only in held-out partitions are also mapped to `Other`.

This procedure reduces dimensional sparsity while preserving partition isolation.

In [26]:
# Learn rare-category mappings from training data only

RARE_CATEGORY_MIN_COUNT = 100


def fit_category_registry(df, categorical_features, min_count=100):
    registry = {}

    for col in categorical_features:
        normalized = df[col].astype("string").fillna("Missing")

        counts = normalized.value_counts(dropna=False)

        retained = set(counts[counts >= min_count].index.astype(str))

        registry[col] = retained

    return registry


category_registry = fit_category_registry(
    X_train_model,
    CATEGORICAL_FEATURES,
    min_count=RARE_CATEGORY_MIN_COUNT,
)

print(
    f"Training-derived category registry "
    f"(minimum count = {RARE_CATEGORY_MIN_COUNT})"
)
print("-" * 80)

for col in CATEGORICAL_FEATURES:
    original_n = X_train_model[col].astype("string").fillna("Missing").nunique()

    retained_n = len(category_registry[col])

    print(f"{col:<40} " f"original={original_n:>3} | " f"retained={retained_n:>3}")

Training-derived category registry (minimum count = 100)
--------------------------------------------------------------------------------
race                                     original=  6 | retained=  6
gender                                   original=  3 | retained=  2
age                                      original= 10 | retained= 10
payer_code                               original= 17 | retained= 13
medical_specialty                        original= 70 | retained= 24
max_glu_serum                            original=  4 | retained=  4
A1Cresult                                original=  4 | retained=  4
metformin                                original=  4 | retained=  4
repaglinide                              original=  4 | retained=  2
nateglinide                              original=  4 | retained=  2
chlorpropamide                           original=  4 | retained=  1
glimepiride                              original=  4 | retained=  4
acetohexamide                     

In [27]:
# Apply training-derived rare-category mappings


def apply_category_registry(df, categorical_features, registry):
    result = df.copy()

    for col in categorical_features:
        normalized = result[col].astype("string").fillna("Missing")

        result[col] = normalized.where(
            normalized.isin(registry[col]),
            "Other",
        )

    return result


X_train_consolidated = apply_category_registry(
    X_train_model,
    CATEGORICAL_FEATURES,
    category_registry,
)

X_validation_consolidated = apply_category_registry(
    X_validation_model,
    CATEGORICAL_FEATURES,
    category_registry,
)

X_test_consolidated = apply_category_registry(
    X_test_model,
    CATEGORICAL_FEATURES,
    category_registry,
)

assert list(X_train_consolidated.columns) == list(X_validation_consolidated.columns)
assert list(X_train_consolidated.columns) == list(X_test_consolidated.columns)

print("✓ Training-derived category mappings applied to all partitions.")

✓ Training-derived category mappings applied to all partitions.


In [28]:
# Audit categorical dimensionality after consolidation

category_consolidation_audit = []

for col in CATEGORICAL_FEATURES:
    before = X_train_model[col].astype("string").fillna("Missing").nunique()

    after = X_train_consolidated[col].nunique(dropna=False)

    other_n = (X_train_consolidated[col] == "Other").sum()

    category_consolidation_audit.append(
        {
            "feature": col,
            "categories_before": before,
            "categories_after": after,
            "other_n": other_n,
            "other_pct": other_n / len(X_train_consolidated) * 100,
        }
    )

category_consolidation_audit = pd.DataFrame(category_consolidation_audit)

category_consolidation_audit["other_pct"] = category_consolidation_audit[
    "other_pct"
].round(2)

category_consolidation_audit = category_consolidation_audit.sort_values(
    ["categories_before", "feature"],
    ascending=[False, True],
).reset_index(drop=True)

category_consolidation_audit

,feature,categories_before,categories_after,other_n,other_pct
0,medical_specialty,70,25,1031,1.44
1,diag_2_family,19,19,85,0.12
2,diag_3_family,19,19,75,0.10
3,diag_1_family,18,17,51,0.07
4,admission_source_description,17,10,116,0.16
5,payer_code,17,14,252,0.35
6,age,10,10,0,0.00
7,admission_type_description,8,7,23,0.03
8,race,6,6,1063,1.49
9,A1Cresult,4,4,0,0.00


## 4L — Reproducible Preprocessing Pipeline

The final preprocessing architecture uses a `ColumnTransformer` so numeric and categorical variables can be transformed through separate, reproducible pathways.

### Numeric pathway

Numeric variables receive median imputation followed by standardization. Median imputation provides robustness to skewed quantitative distributions, while standardization supports scale-sensitive estimators such as regularized logistic regression.

### Categorical pathway

Categorical variables receive most-frequent imputation followed by one-hot encoding. The encoder is configured to tolerate previously unseen categories so held-out observations cannot cause transformation failures.

Critically, the preprocessing object is fit **only on the training partition**. Validation and test partitions are transformed using the fitted training parameters without refitting.

In [29]:
# Construct numeric and categorical preprocessing pipelines

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

print(preprocessor)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['time_in_hospital', 'num_lab_procedures',
                                  'num_procedures', 'num_medications',
                                  'number_outpatient', 'number_emergency',
                                  'number_inpatient', 'number_diagnoses',
                                  'prior_utilization_total',
                                  'log1p_number_outpatient',
                                  'log1p_numb...
                                  'A1Cresult', 'metformin', 'repaglinide',
                                  'nateglinide', 'chlorpropamide',
                                  'glimepiride', 'acetohexamide', 'glipizide',
                                  'glybur

In [30]:
# Fit preprocessing exclusively on training data

preprocessor.fit(X_train_consolidated)

X_train_ready = preprocessor.transform(X_train_consolidated)

X_validation_ready = preprocessor.transform(X_validation_consolidated)

X_test_ready = preprocessor.transform(X_test_consolidated)

print("Model-ready matrix dimensions")
print("-" * 60)

print(f"Train:      {X_train_ready.shape}")
print(f"Validation: {X_validation_ready.shape}")
print(f"Test:       {X_test_ready.shape}")

assert X_train_ready.shape[1] == X_validation_ready.shape[1]
assert X_train_ready.shape[1] == X_test_ready.shape[1]

print("\n✓ Identical transformed feature space across all partitions.")
print("✓ Preprocessor was fit exclusively on training data.")

Model-ready matrix dimensions
------------------------------------------------------------
Train:      (71520, 213)
Validation: (15027, 213)
Test:       (15219, 213)

✓ Identical transformed feature space across all partitions.
✓ Preprocessor was fit exclusively on training data.


## 4M — Transformed Feature Registry

After fitting the preprocessing pipeline on the training partition, the transformed design matrix can be mapped back to human-readable feature names.

Maintaining this registry is important for downstream model interpretation, coefficient analysis, feature-importance reporting, debugging, and reproducibility. It also provides a direct audit of the dimensional expansion introduced by categorical encoding.

The transformed feature registry is derived exclusively from the fitted training preprocessing pipeline.

In [31]:
# Recover names for every transformed modeling feature

transformed_feature_names = preprocessor.get_feature_names_out()

print("Transformed feature registry")
print("-" * 70)

print(f"Raw primary predictors:     {X_train_model.shape[1]:>5}")
print(f"Numeric transformed:        {len(NUMERIC_FEATURES):>5}")
print(
    f"Categorical transformed:    {len(transformed_feature_names) - len(NUMERIC_FEATURES):>5}"
)
print(f"Total transformed features: {len(transformed_feature_names):>5}")

assert len(transformed_feature_names) == X_train_ready.shape[1]
assert len(set(transformed_feature_names)) == len(transformed_feature_names)

print("\n✓ Feature names align exactly with transformed matrix columns.")
print("✓ All transformed feature names are unique.")

Transformed feature registry
----------------------------------------------------------------------
Raw primary predictors:        48
Numeric transformed:           13
Categorical transformed:      200
Total transformed features:   213

✓ Feature names align exactly with transformed matrix columns.
✓ All transformed feature names are unique.


In [32]:
# Construct a machine-readable transformed feature registry

transformed_feature_registry = pd.DataFrame(
    {
        "feature_index": range(len(transformed_feature_names)),
        "transformed_feature": transformed_feature_names,
    }
)

transformed_feature_registry["feature_type"] = (
    transformed_feature_registry["transformed_feature"].str.split("__", n=1).str[0]
)

transformed_feature_registry.head(25)

,feature_index,transformed_feature,feature_type
0,0,numeric__time_in_hospital,numeric
1,1,numeric__num_lab_procedures,numeric
2,2,numeric__num_procedures,numeric
3,3,numeric__num_medications,numeric
4,4,numeric__number_outpatient,numeric
5,5,numeric__number_emergency,numeric
6,6,numeric__number_inpatient,numeric
7,7,numeric__number_diagnoses,numeric
8,8,numeric__prior_utilization_total,numeric
9,9,numeric__log1p_number_outpatient,numeric


## 4N — Matrix Integrity and Sparsity Audit

Before persisting the modeling artifacts, the transformed matrices are subjected to structural and numerical integrity checks.

The audit verifies that:

- training, validation, and test partitions share an identical transformed feature space;
- transformed matrices contain no missing or infinite values;
- target vectors remain aligned with their corresponding encounter matrices;
- the preprocessing pipeline preserves the expected row counts; and
- the encoded representation retains an efficient sparse structure where applicable.

These checks establish a reproducible boundary between feature engineering and predictive modeling.

In [33]:
# Audit transformed matrices before model development

from scipy import sparse

matrix_registry = {
    "train": (X_train_ready, y_train),
    "validation": (X_validation_ready, y_validation),
    "test": (X_test_ready, y_test),
}

print("Model-ready matrix integrity audit")
print("-" * 75)

for name, (X_matrix, y_vector) in matrix_registry.items():

    is_sparse = sparse.issparse(X_matrix)

    if is_sparse:
        data_values = X_matrix.data
        nnz = X_matrix.nnz
        total_cells = X_matrix.shape[0] * X_matrix.shape[1]
        density = nnz / total_cells
    else:
        data_values = np.asarray(X_matrix).ravel()
        nnz = np.count_nonzero(data_values)
        total_cells = X_matrix.size
        density = nnz / total_cells

    assert X_matrix.shape[0] == len(y_vector)
    assert X_matrix.shape[1] == len(transformed_feature_names)

    assert not np.isnan(data_values).any()
    assert np.isfinite(data_values).all()

    print(
        f"{name.title():<12} "
        f"shape={str(X_matrix.shape):<18} "
        f"sparse={str(is_sparse):<5} "
        f"density={density:.4f}"
    )

print("\n✓ Row counts remain aligned with target vectors.")
print("✓ No NaN values exist in transformed matrices.")
print("✓ No infinite values exist in transformed matrices.")
print("✓ All partitions share the same transformed feature space.")

Model-ready matrix integrity audit
---------------------------------------------------------------------------
Train        shape=(71520, 213)       sparse=True  density=0.2254
Validation   shape=(15027, 213)       sparse=True  density=0.2254
Test         shape=(15219, 213)       sparse=True  density=0.2254

✓ Row counts remain aligned with target vectors.
✓ No NaN values exist in transformed matrices.
✓ No infinite values exist in transformed matrices.
✓ All partitions share the same transformed feature space.


In [34]:
# Verify target integrity after feature construction

target_audit = pd.DataFrame(
    {
        "partition": ["train", "validation", "test"],
        "n": [
            len(y_train),
            len(y_validation),
            len(y_test),
        ],
        "positive_n": [
            int(y_train.sum()),
            int(y_validation.sum()),
            int(y_test.sum()),
        ],
    }
)

target_audit["positive_rate_pct"] = (
    target_audit["positive_n"] / target_audit["n"] * 100
).round(2)

target_audit

,partition,n,positive_n,positive_rate_pct
0,train,71520,8021,11.22
1,validation,15027,1656,11.02
2,test,15219,1680,11.04


In [35]:
assert set(pd.unique(y_train)).issubset({0, 1})
assert set(pd.unique(y_validation)).issubset({0, 1})
assert set(pd.unique(y_test)).issubset({0, 1})

print("✓ Binary target integrity confirmed across all partitions.")

✓ Binary target integrity confirmed across all partitions.


## 4O — Persist Modeling Artifacts

The fitted preprocessing pipeline and associated metadata are persisted for use by downstream modeling notebooks.

Persisting the fitted training-derived transformer ensures that future models operate on the same preprocessing specification without independently reconstructing feature-engineering decisions.

The saved artifacts include:

- the fitted preprocessing transformer;
- the transformed feature-name registry;
- the semantic numeric and categorical feature definitions;
- the training-derived rare-category registry; and
- configuration metadata required to reproduce the final modeling feature space.

The validation and test partitions remain held out from all fitting operations.

In [36]:
# Prepare reproducible modeling-artifact directories

import json
import joblib

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Models directory:  {MODELS_DIR}")
print(f"Reports directory: {REPORTS_DIR}")

Models directory:  c:\Users\amohe.000\Downloads\healthcare-readmission-analytics\models
Reports directory: c:\Users\amohe.000\Downloads\healthcare-readmission-analytics\reports


In [37]:
# Persist fitted preprocessing transformer

preprocessor_path = MODELS_DIR / "preprocessor.joblib"

joblib.dump(
    preprocessor,
    preprocessor_path,
)

print(f"Saved fitted preprocessor:")
print(preprocessor_path)

Saved fitted preprocessor:
c:\Users\amohe.000\Downloads\healthcare-readmission-analytics\models\preprocessor.joblib


In [38]:
# Persist transformed feature registry

feature_registry_path = REPORTS_DIR / "transformed_feature_registry.csv"

transformed_feature_registry.to_csv(
    feature_registry_path,
    index=False,
)

print(f"Saved transformed feature registry:")
print(feature_registry_path)

Saved transformed feature registry:
c:\Users\amohe.000\Downloads\healthcare-readmission-analytics\reports\transformed_feature_registry.csv


In [39]:
# Persist feature-engineering configuration and provenance metadata

feature_metadata = {
    "random_state": RANDOM_STATE,
    "target": TARGET,
    "rare_category_min_count": RARE_CATEGORY_MIN_COUNT,
    "n_train": len(X_train_model),
    "n_validation": len(X_validation_model),
    "n_test": len(X_test_model),
    "primary_predictor_count": X_train_model.shape[1],
    "numeric_feature_count": len(NUMERIC_FEATURES),
    "categorical_feature_count": len(CATEGORICAL_FEATURES),
    "transformed_feature_count": len(transformed_feature_names),
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "primary_model_exclusions": PRIMARY_MODEL_EXCLUSIONS,
    "diagnosis_family_features": DIAGNOSIS_FAMILY_COLUMNS,
    "transformed_features": transformed_feature_names.tolist(),
}

metadata_path = REPORTS_DIR / "feature_engineering_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        feature_metadata,
        file,
        indent=2,
    )

print(f"Saved feature-engineering metadata:")
print(metadata_path)

Saved feature-engineering metadata:
c:\Users\amohe.000\Downloads\healthcare-readmission-analytics\reports\feature_engineering_metadata.json


In [40]:
# Persist training-derived category registry

serializable_category_registry = {
    feature: sorted(str(value) for value in retained_values)
    for feature, retained_values in category_registry.items()
}

category_registry_path = REPORTS_DIR / "category_registry.json"

with open(category_registry_path, "w", encoding="utf-8") as file:
    json.dump(
        serializable_category_registry,
        file,
        indent=2,
    )

print(f"Saved category registry:")
print(category_registry_path)

Saved category registry:
c:\Users\amohe.000\Downloads\healthcare-readmission-analytics\reports\category_registry.json


## 4P — Final Feature-Engineering Reproducibility Audit

The feature-engineering stage concludes with an explicit reproducibility audit.

A successful audit confirms that:

1. patient partitions remain mutually exclusive;
2. outcome and identifier variables are excluded from predictor matrices;
3. diagnosis-family representations replace raw high-cardinality diagnosis codes;
4. discharge disposition is excluded from the primary prediction model;
5. rare-category mappings are derived exclusively from training data;
6. preprocessing parameters are fitted exclusively on the training partition;
7. validation and test observations are transformed without refitting;
8. all transformed matrices have identical feature dimensionality;
9. transformed matrices contain only finite, non-missing values; and
10. persisted artifacts provide the exact preprocessing specification required for downstream modeling.

Passing these checks establishes the final modeling dataset and closes the feature-engineering stage.

In [41]:
# Final automated feature-engineering audit

print("FINAL FEATURE-ENGINEERING AUDIT")
print("=" * 75)

# Partition integrity
assert train_patients.isdisjoint(validation_patients)
assert train_patients.isdisjoint(test_patients)
assert validation_patients.isdisjoint(test_patients)

print("✓ Patient partitions are mutually exclusive.")

# Predictor exclusion integrity
for excluded_col in MANDATORY_EXCLUSIONS:
    assert excluded_col not in X_train_model.columns

print("✓ Mandatory identifiers/outcome fields are excluded.")

# Raw diagnosis exclusion
for col in ["diag_1", "diag_2", "diag_3"]:
    assert col not in X_train_model.columns

print("✓ Raw high-cardinality diagnosis codes are excluded.")

# Diagnosis-family presence
for col in DIAGNOSIS_FAMILY_COLUMNS:
    assert col in X_train_model.columns

print("✓ Consolidated diagnosis-family predictors are present.")

# Discharge disposition exclusion
assert "discharge_disposition_description" not in X_train_model.columns

print("✓ Discharge disposition is excluded from the primary model.")

# Semantic taxonomy
assert set(NUMERIC_FEATURES).isdisjoint(CATEGORICAL_FEATURES)

assert set(NUMERIC_FEATURES) | set(CATEGORICAL_FEATURES) == set(X_train_model.columns)

print("✓ Semantic feature taxonomy is complete and non-overlapping.")

# Matrix dimensionality
assert X_train_ready.shape[1] == len(transformed_feature_names)
assert X_validation_ready.shape[1] == len(transformed_feature_names)
assert X_test_ready.shape[1] == len(transformed_feature_names)

print("✓ Transformed feature dimensionality is consistent.")

# Row alignment
assert X_train_ready.shape[0] == len(y_train)
assert X_validation_ready.shape[0] == len(y_validation)
assert X_test_ready.shape[0] == len(y_test)

print("✓ Predictor and target rows remain aligned.")

# Artifact existence
for path in [
    preprocessor_path,
    feature_registry_path,
    metadata_path,
    category_registry_path,
]:
    assert path.exists(), f"Missing artifact: {path}"

print("✓ Required preprocessing artifacts were persisted.")

print("\n" + "=" * 75)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 75)

print(f"Primary predictors:      {X_train_model.shape[1]}")
print(f"Numeric predictors:      {len(NUMERIC_FEATURES)}")
print(f"Categorical predictors:  {len(CATEGORICAL_FEATURES)}")
print(f"Transformed features:    {len(transformed_feature_names)}")
print(f"Training encounters:     {len(y_train):,}")
print(f"Validation encounters:   {len(y_validation):,}")
print(f"Test encounters:         {len(y_test):,}")

FINAL FEATURE-ENGINEERING AUDIT
✓ Patient partitions are mutually exclusive.
✓ Mandatory identifiers/outcome fields are excluded.
✓ Raw high-cardinality diagnosis codes are excluded.
✓ Consolidated diagnosis-family predictors are present.
✓ Discharge disposition is excluded from the primary model.
✓ Semantic feature taxonomy is complete and non-overlapping.
✓ Transformed feature dimensionality is consistent.
✓ Predictor and target rows remain aligned.
✓ Required preprocessing artifacts were persisted.

FEATURE ENGINEERING COMPLETE
Primary predictors:      48
Numeric predictors:      13
Categorical predictors:  35
Transformed features:    213
Training encounters:     71,520
Validation encounters:   15,027
Test encounters:         15,219


## 4Q — Feature Engineering Conclusions and Modeling Handoff

The feature-engineering stage produced a reproducible, leakage-controlled modeling representation for 30-day hospital readmission prediction.

The final primary predictor set contains 48 semantically defined variables. High-cardinality raw diagnosis codes were replaced with clinically interpretable diagnosis-family representations, prior healthcare utilization was represented through both aggregate and skew-adjusted measures, and discharge disposition was excluded from the primary model to maintain a defensible prediction timestamp.

Categorical sparsity was controlled through training-derived rare-category consolidation, after which a training-fitted preprocessing pipeline generated a consistent transformed feature space across the training, validation, and test partitions. All transformation parameters were learned exclusively from training observations.

The resulting design matrices contain 213 transformed predictors and preserve the patient-level isolation established during preprocessing. The fitted transformer, category registry, feature-name registry, and feature-engineering metadata have been persisted as reproducible modeling artifacts.

The project can therefore proceed to predictive modeling without redefining the feature space or accessing test-set information during model development.

### Next stage

Notebook `05_baseline_modeling.ipynb` will establish the first predictive benchmarks for 30-day readmission. Model evaluation will emphasize performance appropriate for an imbalanced clinical outcome, including discrimination, minority-class detection, calibration, and threshold-dependent operating characteristics.

The held-out test partition will remain untouched during model selection and hyperparameter development.